# cd_01_extract_procore

Pulls every endpoint in `config/endpoints.yml` into **CD_Bronze_Lakehouse**.

This notebook holds no endpoint logic. Auth, pagination, the v2.0 header rule, 429 retry
and watermarking live in `procore_extract.py`; parent-scope expansion lives in
`procore_scope.py`. Adding an endpoint is a YAML entry, not a change here.

**Attach this notebook to `CD_Bronze_Lakehouse` as its default lakehouse before running.**

Prerequisites:
- `Files/lib/` holds `fabric_common.py`, `procore_scope.py`, `procore_extract.py`
- `Files/config/endpoints.yml`
- Key Vault holds `PROCORE_CLIENT_ID`, `PROCORE_CLIENT_SECRET`, `PROCORE_COMPANY_ID`,
  and `PROCORE_KEYVAULT_URL` is set on the environment

In [ ]:
import sys, os

# The shared library ships as Files/ in the lakehouse rather than being pasted into the
# notebook. One copy, one place to fix a bug.
sys.path.insert(0, "/lakehouse/default/Files/lib")

import requests

import fabric_common as fc
import procore_scope as ps
import procore_extract as px
import watermark as wm

CONFIG = "/lakehouse/default/Files/config/endpoints.yml"

batch_id = fc.new_batch_id()
print(f"batch {batch_id}")

In [ ]:
# Registry is validated on load: duplicate names, duplicate destination tables, a parent
# pointing at an endpoint that does not exist, or a parent cycle all raise here rather
# than surfacing hours later as an empty table.
endpoints = ps.load_registry(CONFIG)
ordered = ps.resolution_order(endpoints)   # parents before their children

print(f"{len(ordered)} endpoints")
for scope in ("company", "project", "parent"):
    names = [e.name for e in ordered if e.scope == scope]
    print(f"  {scope:<8} {len(names):>2}  {', '.join(names[:6])}{' ...' if len(names) > 6 else ''}")

In [ ]:
settings = px.load_settings()
session = requests.Session()
token = px.fetch_token(settings, session)
print("authenticated")

# Active projects only. The existing notebooks loop EVERY project on every run; most are
# closed, and Procore's rate limits are high but real. Jul 23 warehouse review.
projects = list(px.iter_active_projects(session, settings, token))
project_ids = [p["id"] for p in projects if p.get("id") is not None]
print(f"{len(project_ids)} active projects")

In [ ]:
from datetime import timezone

# Records kept per endpoint so parent-scoped endpoints can read their parents' ids.
fetched: dict[str, list[dict]] = {}
summary = []

for ep in ordered:
    parent_ids = None
    if ep.parent:
        parent_ids = ps.collect_parent_ids(fetched.get(ep.parent.endpoint, []), ep.parent)
        if not parent_ids:
            # Not an error: a company with no prime contracts has no line items either.
            print(f"  {ep.name:<32} skipped - parent '{ep.parent.endpoint}' returned nothing")
            summary.append((ep.name, 0, "skipped"))
            continue

    # Watermark is read BEFORE the pull and written only after it succeeds.
    since = wm.read_since(spark, ep.bronze_table, ep.name) if ep.incremental else None

    headers = px.build_headers(token, settings.company_id, ep)
    params = px.watermark_params(ep, since) if ep.incremental else {}
    if ep.scope == "company":
        params = {**params, "company_id": settings.company_id}

    records, rows = [], []
    ingested_at = fc.utc_now()
    for path, project_id in ps.expand_paths(ep, settings.company_id, project_ids, parent_ids):
        for record in px.iter_records(session, settings.base_url, path, headers, params=params):
            records.append(record)
            rows.append({
                **px.to_bronze_row(record, ep, project_id, ingested_at),
                "_batch_id": batch_id,
                "_row_hash": fc.row_hash(record),
            })

    fetched[ep.name] = records

    if rows:
        df = spark.createDataFrame(rows)
        # MERGE on the natural key, not DROP + append: re-running is a no-op, so the
        # deliberate one-hour watermark overlap cannot duplicate rows.
        fc.merge_delta(spark, df, ep.bronze_table, ["_key"])

        high = wm.high_water(records, "updated_at")
        if ep.incremental and high:
            wm.write_watermark(spark, ep.bronze_table, ep.name, high, batch_id)

    fc.log_run(spark, batch_id, "extract_procore", ep.bronze_table, len(rows))
    summary.append((ep.name, len(rows), "incremental" if since else "full"))
    print(f"  {ep.name:<32} {len(rows):>7} rows  ({summary[-1][2]})")

In [ ]:
total = sum(n for _, n, _ in summary)
empty = [name for name, n, mode in summary if n == 0 and mode != "skipped"]

print(f"\nbatch {batch_id}: {total} rows across {len(summary)} endpoints")
if empty:
    # Worth surfacing rather than burying: on a full reload an empty result usually means
    # a permission gap or a tool Affect does not use, not genuinely zero records.
    print(f"returned nothing: {', '.join(empty)}")